# 01. Synthetic Data Generation

This notebook generates realistic synthetic student data for training and demo purposes.

## Data Generation Logic

### Features Generated:
| Feature | Distribution | Parameters | Range |
|---------|-------------|------------|-------|
| attendance_percentage | Normal | μ=75, σ=20 | 0-100 |
| assignment_submission_rate | Normal | μ=70, σ=25 | 0-100 |
| internal_marks | Computed | Correlated with attendance & assignment | 0-100 |
| semester | Uniform | Random integer | 1-8 |

### Risk Score Formula:
```
risk_score = (100 - attendance) * 0.4 + 
             (100 - internal_marks) * 0.4 + 
             (100 - assignment_rate) * 0.2 + noise
```

### Risk Level Thresholds:
- **High Risk**: risk_score >= 55
- **Moderate Risk**: 35 <= risk_score < 55
- **Low Risk**: risk_score < 35

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import json

# Set seeds for reproducibility
fake = Faker()
Faker.seed(42)
np.random.seed(42)
random.seed(42)

## Data Generation Configuration

This config can be saved and reused for generating consistent synthetic data.

In [2]:
# Configuration for synthetic data generation
DATA_GENERATION_CONFIG = {
    'n_students': 500,
    'random_seed': 42,
    'features': {
        'attendance_percentage': {
            'distribution': 'normal',
            'mean': 75,
            'std': 20,
            'clip_min': 0,
            'clip_max': 100
        },
        'assignment_submission_rate': {
            'distribution': 'normal',
            'mean': 70,
            'std': 25,
            'clip_min': 0,
            'clip_max': 100
        },
        'internal_marks': {
            'distribution': 'computed',
            'base': 50,
            'attendance_weight': 0.3,
            'assignment_weight': 0.2,
            'noise_std': 15,
            'clip_min': 0,
            'clip_max': 100
        },
        'semester': {
            'distribution': 'uniform_int',
            'min': 1,
            'max': 8
        }
    },
    'risk_scoring': {
        'weights': {
            'attendance': 0.4,
            'internal_marks': 0.4,
            'assignment_rate': 0.2
        },
        'noise_std': 10,
        'thresholds': {
            'high_risk': 55,
            'moderate_risk': 35
        }
    }
}

# Save config for future use
with open('../data/raw/data_generation_config.json', 'w') as f:
    json.dump(DATA_GENERATION_CONFIG, f, indent=2)

print("Configuration saved to ../data/raw/data_generation_config.json")

Configuration saved to ../data/raw/data_generation_config.json


## Data Generation Function

In [3]:
def generate_student_data(n_students=500, config=None):
    """
    Generate synthetic student academic data with realistic correlations.
    
    The generation logic simulates real-world patterns:
    - Students with higher attendance tend to have higher marks
    - Assignment submission correlates with overall performance
    - Risk is computed from multiple weighted factors
    
    Parameters:
    -----------
    n_students : int
        Number of student records to generate
    config : dict, optional
        Configuration dictionary. Uses DATA_GENERATION_CONFIG if None.
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with student records
    """
    
    if config is None:
        config = DATA_GENERATION_CONFIG
    
    data = []
    
    for i in range(n_students):
        # Generate attendance (key predictor)
        att_config = config['features']['attendance_percentage']
        attendance = np.clip(
            np.random.normal(att_config['mean'], att_config['std']),
            att_config['clip_min'],
            att_config['clip_max']
        )
        
        # Generate assignment submission rate
        assign_config = config['features']['assignment_submission_rate']
        assignment_rate = np.clip(
            np.random.normal(assign_config['mean'], assign_config['std']),
            assign_config['clip_min'],
            assign_config['clip_max']
        )
        
        # Compute internal marks with correlation to attendance & assignment
        # This simulates real pattern: engaged students perform better
        marks_config = config['features']['internal_marks']
        base_marks = (
            marks_config['base'] +
            marks_config['attendance_weight'] * attendance +
            marks_config['assignment_weight'] * assignment_rate
        )
        internal_marks = np.clip(
            base_marks + np.random.normal(0, marks_config['noise_std']),
            marks_config['clip_min'],
            marks_config['clip_max']
        )
        
        # Random semester
        sem_config = config['features']['semester']
        semester = random.randint(sem_config['min'], sem_config['max'])
        
        # Calculate risk score
        risk_config = config['risk_scoring']
        risk_score = (
            (100 - attendance) * risk_config['weights']['attendance'] +
            (100 - internal_marks) * risk_config['weights']['internal_marks'] +
            (100 - assignment_rate) * risk_config['weights']['assignment_rate']
        )
        
        # Add noise to risk score
        risk_score += np.random.normal(0, risk_config['noise_std'])
        
        # Classify risk level based on thresholds
        thresholds = risk_config['thresholds']
        if risk_score >= thresholds['high_risk']:
            risk_level = 'High'
        elif risk_score >= thresholds['moderate_risk']:
            risk_level = 'Moderate'
        else:
            risk_level = 'Low'
        
        data.append({
            'student_id': f'STU{i+1:04d}',
            'student_name': fake.name(),
            'attendance_percentage': round(attendance, 1),
            'internal_marks': round(internal_marks, 1),
            'assignment_submission_rate': round(assignment_rate, 1),
            'semester': semester,
            'risk_score': round(risk_score, 2),
            'risk_level': risk_level
        })
    
    return pd.DataFrame(data)

# Generate dataset
df = generate_student_data(500)
print(f"Generated {len(df)} student records")
df.head()

Generated 500 student records


,student_id,student_name,attendance_percentage,internal_marks,assignment_submission_rate,semester,risk_score,risk_level
0,STU0001,Allison Hill,84.9,98.5,66.5,2,28.55,Low
1,STU0002,Noah Rhodes,70.3,100.0,64.1,1,26.72,Low
2,STU0003,Angie Henderson,65.6,79.4,83.6,5,20.61,Low
3,STU0004,Daniel Wagner,79.8,52.5,22.2,4,37.00,Moderate
4,STU0005,Cristian Santos,54.7,68.4,77.9,4,21.06,Low


In [4]:
# Save the generated dataset
df.to_csv('../data/raw/synthetic_students.csv', index=False)
print(f"\nDataset saved to: ../data/raw/synthetic_students.csv")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")


Dataset saved to: ../data/raw/synthetic_students.csv
Shape: 500 rows x 8 columns


## Data Quality Summary

In [5]:
# Display data statistics
print("\n" + "="*60)
print("SYNTHETIC DATA SUMMARY")
print("="*60)

print("\n1. Risk Level Distribution:")
risk_dist = df['risk_level'].value_counts()
for level, count in risk_dist.items():
    pct = count / len(df) * 100
    print(f"   {level}: {count} ({pct:.1f}%)")

print("\n2. Feature Statistics:")
print(df[['attendance_percentage', 'internal_marks', 'assignment_submission_rate']].describe().round(2))

print("\n3. Sample Records per Risk Level:")
for level in ['Low', 'Moderate', 'High']:
    sample = df[df['risk_level'] == level].iloc[0]
    print(f"\n   {level} Risk:")
    print(f"      Attendance: {sample['attendance_percentage']}%")
    print(f"      Internal Marks: {sample['internal_marks']}")
    print(f"      Assignment Rate: {sample['assignment_submission_rate']}%")
    print(f"      Risk Score: {sample['risk_score']}")


SYNTHETIC DATA SUMMARY

1. Risk Level Distribution:
   Low: 400 (80.0%)
   Moderate: 92 (18.4%)
   High: 8 (1.6%)

2. Feature Statistics:
       attendance_percentage  internal_marks  assignment_submission_rate
count                 500.00          500.00                      500.00
mean                   74.69           85.14                       69.85
std                    16.91           13.45                       22.12
min                    22.00           32.00                        0.00
25%                    62.70           76.10                       54.48
50%                    75.95           87.30                       71.55
75%                    87.65           97.25                       87.48
max                   100.00          100.00                      100.00

3. Sample Records per Risk Level:

   Low Risk:
      Attendance: 84.9%
      Internal Marks: 98.5
      Assignment Rate: 66.5%
      Risk Score: 28.55

   Moderate Risk:
      Attendance: 79.8%
      In

## Feature Insights for Future Use

In [6]:
# Key insights about feature relationships
print("\nFEATURE INSIGHTS (for documentation):")
print("\n1. ATTENDANCE PERCENTAGE:")
print("   - Primary predictor of student success")
print("   - Directly impacts internal marks (correlation ~0.5)")
print("   - Low attendance (<60%) strongly indicates high risk")

print("\n2. INTERNAL MARKS:")
print("   - Computed from attendance + assignment + noise")
print("   - Represents cumulative academic performance")
print("   - Weight: 40% in risk calculation")

print("\n3. ASSIGNMENT SUBMISSION RATE:")
print("   - Indicates student engagement")
print("   - Early warning signal (drops before marks)")
print("   - Weight: 20% in risk calculation")

print("\n4. SEMESTER:")
print("   - Included for potential session-based analysis")
print("   - Can track if risk patterns vary by academic year")

# Save insights
insights = {
    'feature_descriptions': {
        'attendance_percentage': 'Student attendance rate (0-100%). Primary predictor. Low attendance <60% indicates high risk.',
        'internal_marks': 'Average internal assessment marks (0-100). Computed from attendance + assignment. 40% weight in risk.',
        'assignment_submission_rate': 'Assignment submission percentage (0-100). Early engagement indicator. 20% weight in risk.',
        'semester': 'Academic semester (1-8). For session-based analysis.'
    },
    'risk_formula': 'risk_score = (100-attendance)*0.4 + (100-internal_marks)*0.4 + (100-assignment)*0.2 + noise',
    'thresholds': {
        'High Risk': 'risk_score >= 55',
        'Moderate Risk': '35 <= risk_score < 55',
        'Low Risk': 'risk_score < 35'
    },
    'key_patterns': [
        'Attendance correlates with marks (engaged students perform better)',
        'Assignment rate is an early warning signal',
        'Risk scores have realistic noise to simulate real-world uncertainty'
    ]
}

with open('../data/raw/feature_insights.json', 'w') as f:
    json.dump(insights, f, indent=2)

print("\nFeature insights saved to: ../data/raw/feature_insights.json")


FEATURE INSIGHTS (for documentation):

1. ATTENDANCE PERCENTAGE:
   - Primary predictor of student success
   - Directly impacts internal marks (correlation ~0.5)
   - Low attendance (<60%) strongly indicates high risk

2. INTERNAL MARKS:
   - Computed from attendance + assignment + noise
   - Represents cumulative academic performance
   - Weight: 40% in risk calculation

3. ASSIGNMENT SUBMISSION RATE:
   - Indicates student engagement
   - Early warning signal (drops before marks)
   - Weight: 20% in risk calculation

4. SEMESTER:
   - Included for potential session-based analysis
   - Can track if risk patterns vary by academic year

Feature insights saved to: ../data/raw/feature_insights.json
